# 08 — Backup Name Movers após knockout causal

Reproduzimos a intervenção da seção 3.4 de arXiv:2211.00593. As três Name Movers primárias são substituídas, somente em `END`, pela média de `hook_result` em pABC condicionada ao grupo exato `(pattern, template_id)` e à posição absoluta. Medimos IO−S, escrita direta IO−S e atenção `END→IO` versus `END→S1` nas oito Backup Name Movers publicadas.

In [1]:
import json
import subprocess
from pathlib import Path

import plotly.express as px
import torch

from ioi_replication.paper import (
    PUBLISHED_CIRCUIT,
    CircuitClass,
    build_paired_dataset,
    cache_grouped_mean_activations,
    measure_backup_name_movers,
)
from ioi_replication.runtime import load_gpt2_small, select_device

SEED = 308
N = 16
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULT_PATH = PROJECT_ROOT / "results/paper/08_backup_name_movers.json"

device = select_device()
model = load_gpt2_small(device)
dataset = build_paired_dataset(model.tokenizer, n=N, seed=SEED)
group_means = cache_grouped_mean_activations(
    model,
    dataset.abc_tokens.to(device),
    dataset.groups,
    activation="result",
)
experiment = measure_backup_name_movers(
    model=model,
    dataset=dataset,
    group_means=group_means,
)

print(f"device={device}; N={N}; grupos={len(dataset.groups)}")
print(
    f"IO−S: {experiment.baseline_logit_difference.mean:+.4f} → "
    f"{experiment.knockout_logit_difference.mean:+.4f} "
    f"(Δ={experiment.delta_logit_difference.mean:+.4f})"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

device=cuda; N=16; grupos=13
IO−S: +3.1144 → +2.9733 (Δ=-0.1410)


## Contribuição basal e compensação

**Contribuição basal** é a DLA ou a margem de atenção no forward intacto. **Compensação** é a mudança pareada `knockout − baseline` depois de remover causalmente as três Name Movers primárias. Uma cabeça pode ter contribuição basal positiva sem aumentar sua atividade após a intervenção; por isso os dois valores permanecem separados.

In [2]:
def summarize(values):
    values = values.detach().float()
    se = values.std(unbiased=True) / values.numel() ** 0.5 if values.numel() > 1 else torch.tensor(0.0)
    return {
        "mean": values.mean().item(),
        "standard_error": se.item(),
        "values": values.cpu().tolist(),
    }

plot_rows = []
head_payload = {}
for (layer, head), metrics in experiment.heads.items():
    label = f"{layer}.{head}"
    baseline_margin = metrics.baseline_attention_io.values - metrics.baseline_attention_s.values
    knockout_margin = metrics.knockout_attention_io.values - metrics.knockout_attention_s.values
    delta_margin = knockout_margin - baseline_margin
    head_payload[label] = {
        "baseline_dla": summarize(metrics.baseline_dla.values),
        "knockout_dla": summarize(metrics.knockout_dla.values),
        "delta_dla": summarize(metrics.delta_dla.values),
        "baseline_attention_io": summarize(metrics.baseline_attention_io.values),
        "knockout_attention_io": summarize(metrics.knockout_attention_io.values),
        "delta_attention_io": summarize(metrics.delta_attention_io.values),
        "baseline_attention_s": summarize(metrics.baseline_attention_s.values),
        "knockout_attention_s": summarize(metrics.knockout_attention_s.values),
        "delta_attention_s": summarize(metrics.delta_attention_s.values),
        "baseline_attention_io_minus_s": summarize(baseline_margin),
        "knockout_attention_io_minus_s": summarize(knockout_margin),
        "delta_attention_io_minus_s": summarize(delta_margin),
    }
    plot_rows.extend([
        {"head": label, "medida": "DLA basal", "valor": metrics.baseline_dla.mean},
        {"head": label, "medida": "Δ DLA após knockout", "valor": metrics.delta_dla.mean},
        {"head": label, "medida": "atenção IO−S basal", "valor": baseline_margin.mean().item()},
        {"head": label, "medida": "Δ atenção IO−S", "valor": delta_margin.mean().item()},
    ])

ranking_dla = sorted(
    ({"head": head, **values["delta_dla"]} for head, values in head_payload.items()),
    key=lambda row: row["mean"],
    reverse=True,
)
ranking_attention = sorted(
    ({"head": head, **values["delta_attention_io_minus_s"]} for head, values in head_payload.items()),
    key=lambda row: row["mean"],
    reverse=True,
)

print("Ranking por Δ DLA:")
for row in ranking_dla:
    print(f"  {row['head']}: {row['mean']:+.4f} ± {row['standard_error']:.4f}")
print("Ranking por Δ atenção END→IO menos END→S1:")
for row in ranking_attention:
    print(f"  {row['head']}: {row['mean']:+.4f} ± {row['standard_error']:.4f}")

Ranking por Δ DLA:
  10.2: +0.5378 ± 0.1173
  10.10: +0.4755 ± 0.0982
  11.2: +0.2097 ± 0.1725
  10.6: +0.1940 ± 0.0289
  10.1: +0.1435 ± 0.0414
  11.9: +0.0852 ± 0.0210
  9.7: +0.0051 ± 0.0017
  9.0: +0.0014 ± 0.0024
Ranking por Δ atenção END→IO menos END→S1:
  10.2: +0.2181 ± 0.0415
  10.10: +0.1949 ± 0.0320
  10.1: +0.1448 ± 0.0305
  10.6: +0.1250 ± 0.0174
  11.9: +0.0777 ± 0.0148
  11.2: +0.0519 ± 0.0272
  9.7: +0.0000 ± 0.0000
  9.0: +0.0000 ± 0.0000


In [3]:
dla_rows = [row for row in plot_rows if "DLA" in row["medida"]]
attention_rows = [row for row in plot_rows if "atenção" in row["medida"]]

fig_dla = px.bar(
    dla_rows,
    x="head",
    y="valor",
    color="medida",
    barmode="group",
    title="Backup Name Movers: escrita IO−S basal e compensação causal",
    labels={"valor": "DLA IO−S em END", "head": "cabeça"},
)
fig_dla.update_layout(width=950, height=480)
fig_dla.show()

fig_attention = px.bar(
    attention_rows,
    x="head",
    y="valor",
    color="medida",
    barmode="group",
    title="Backup Name Movers: margem de atenção END→IO menos END→S1",
    labels={"valor": "atenção IO−S", "head": "cabeça"},
)
fig_attention.update_layout(width=950, height=480)
fig_attention.show()

In [4]:
def estimate_payload(estimate):
    return {
        "mean": estimate.mean,
        "standard_error": estimate.standard_error,
        "values": estimate.values.detach().float().cpu().tolist(),
    }

try:
    git_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = None

model_dtype = str(next(model.parameters()).dtype).removeprefix("torch.")
gpu_name = torch.cuda.get_device_name(device) if device.type == "cuda" else None
artifact = {
    "paper": "arXiv:2211.00593",
    "config": {
        "model_name": "gpt2",
        "seed": SEED,
        "n_examples": N,
        "dataset_sha256": dataset.dataset_sha256,
        "git_commit": git_commit,
        "dtype": model_dtype,
        "device": device.type,
        "gpu": gpu_name,
        "n_groups": len(dataset.groups),
        "mean_ablation": "pABC exact (pattern, template_id), absolute position, hook_result",
        "knocked_out_primary_heads": [
            f"{node.layer}.{node.head}"
            for node in PUBLISHED_CIRCUIT[CircuitClass.NAME_MOVERS]
        ],
    },
    "logit_difference": {
        "baseline": estimate_payload(experiment.baseline_logit_difference),
        "primary_name_mover_knockout": estimate_payload(experiment.knockout_logit_difference),
        "delta": estimate_payload(experiment.delta_logit_difference),
    },
    "backup_heads": head_payload,
    "ranking_by_delta_dla": ranking_dla,
    "ranking_by_delta_attention_io_minus_s": ranking_attention,
    "interpretation": {
        "baseline": "Contribuição basal no forward intacto.",
        "compensation": "Mudança pareada após knockout; não é contribuição basal.",
        "attention_s_position": "S1",
    },
}
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print(f"Resultado salvo em {RESULT_PATH.relative_to(PROJECT_ROOT)}")

Resultado salvo em results\paper\08_backup_name_movers.json


## Interpretação e limites

O ranking causal usa a mudança pareada após a intervenção. Ele responde quais Backup Name Movers mudam sua escrita ou atenção quando as Name Movers primárias deixam de escrever em `END`. O ranking basal responde outra pergunta: quais cabeças já contribuíam no modelo intacto. Não inferimos compensação somente de uma contribuição basal alta.

Com `N=16`, as incertezas ainda são amplas e o lote é muito menor que o usado no artigo. Os números servem como smoke experiment reproduzível e como teste qualitativo do mecanismo, não como estimativa precisa dos efeitos populacionais.